# Lessons 11–16
Quick RAG revision, function calling, first agentic steps, agentic loop, ToyAIKit, and other frameworks.

### Lesson 12 Quick RAG Revision (Optional)
Rebuilding the RAG pipeline

In [30]:
# Load environment variables and create the OpenAI client pointing to Gemini's OpenAI-compatible endpoint

import os # Needed to read the Gemini API key from environment variables

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI(
    # Fetch the loaded Gemini key
    api_key=os.getenv("GEMINI_API_KEY"),
    
    # Reroute standard OpenAI traffic to Google's servers
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

print("Gemini-compatible OpenAI client created.")

Gemini-compatible OpenAI client created.


In [31]:
# Load the FAQ data and build the search index

from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

print("FAQ data loaded and search index built.")

FAQ data loaded and search index built.


In [32]:
# Create the RAG assistant with instructions, index, and LLM client

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

print("RAG assistant created.")

RAG assistant created.


In [33]:
# Test the RAG pipeline with a correctly spelled question.
# This should work because lexical search can match "Ollama" in the FAQ index.
print(assistant.rag("How do I run Ollama locally?"))

To run Ollama locally, follow these steps:

1.  **Install Ollama:** Visit [https://ollama.com/download](https://ollama.com/download) and download the installer for your operating system (macOS, Windows, or Linux). For Linux, you can also use the terminal command: `curl -fsSL https://ollama.com/install.sh | sh`.
2.  **Start the model:** Open a terminal and run the command `ollama run llama3`. This will download the model and open a chat interface.
3.  **Verify the server:** To ensure the local server is running, type `curl http://localhost:11434` in your terminal. You should receive a JSON response listing your models.
4.  **Use it in Python (optional):** Install the client via `pip install ollama` to use it in your scripts.

If you encounter a "connection refused" error, you can restart the server by running:
`!nohup ollama serve > nohup.out 2>&1 &`


In [34]:
# Test the fixed pipeline with a typo in the question.
# This will fail because lexical search looks for exact keyword matches, so "Olama" will not match "Ollama" in the index.
print(assistant.rag("How do I run Olama locally?"))

I am sorry, but the provided context does not contain information on how to run Ollama locally.


### Lesson 13 Function Calling
From fixed RAG to tool-using agent

In [62]:
# Setup LLM Model Configuration
MODEL_ID = "gemini-3.1-flash-lite"

print("Model ID configured.")

Model ID configured.


In [36]:
# LLM call with no tools to observe the model's default response behaviour
def llm(instructions, user_prompt, model=MODEL_ID):
    instructions_clean = instructions.strip()

    message_history = [
        {"role": "developer", "content": instructions_clean},
        {"role": "user", "content": user_prompt}
    ]
    
    response = openai_client.chat.completions.create(
        model=model,
        messages=message_history,
     )
    return response.choices[0].message.content

question = "I just discovered the course. Can I join it?"
answer = llm(instructions, question)
print(answer)

Please provide the CONTEXT (the FAQ database) so I can answer your question correctly. Once you provide the text, I will be able to confirm if you can join the course.


In [37]:
# Define a top-level search function for the FAQ index.
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

print("Search function defined.")

Search function defined.


In [38]:
# Describe the search function as a tool schema so the model knows when to call it and what arguments to provide.
# The model only sees this JSON tool description, not the Python function itself.
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.", # Most important field
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"],
            "additionalProperties": False
        }
    }
}
    
print("Search tool schema defined.")

Search tool schema defined.


In [39]:
# Send the tool schema with the request so the model can decide whether to call `search`,
# and return the full API response so we can inspect the tool call and rewritten query.

def llm(instructions, user_prompt, model=MODEL_ID):
    instructions_clean = instructions.strip()

    message_history = [
        {"role": "developer", "content": instructions_clean},
        {"role": "user", "content": user_prompt}
    ]
    
    response = openai_client.chat.completions.create(
        model=model,
        messages=message_history,
        tools=[search_tool],
    )
    return response, message_history # Return the full response and message history for the next tool-calling step.

question = "I just discovered the course. Can I join it?"
answer, message_history = llm(instructions, question) # Keep message_history so we can append the tool call and tool result next.
print(answer)

ChatCompletion(id='3XAuavGSM9awjuMP0NDR8QQ', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='VUd5JXjb', function=Function(arguments='{"query":"can I join the course late"}', name='search'), type='function', extra_content={'google': {'thought_signature': 'EjQKMgEMOdbH0n4QJxBVisI4SzLG74OGLMBW4UrLXnwn3Xlxh1m5tIqmIU+zKg74YaMbnogZ'}})]))], created=1781428446, model='gemini-3.1-flash-lite', object='chat.completion', moderation=None, service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=19, prompt_tokens=111, total_tokens=130, completion_tokens_details=None, prompt_tokens_details=None))


In [40]:
# Display the full model response in a structured, expandable JSON viewer.
# This is useful when we want to inspect the entire API response clearly
# instead of printing one long block of text.
from IPython.display import JSON

# Convert the response object into a plain Python dictionary, then show it
# as formatted JSON in the notebook.
JSON(answer.model_dump())

<IPython.core.display.JSON object>

In [41]:
# In this cell, we inspect the tool call returned by the model above,
# parse its JSON arguments into a Python dict, and run the search function.
# This is good practice for learning how model tool arguments are encoded.

import json

# The course uses the Responses API, where the tool call is in `response.output[0]`.
# We are using Chat Completions, so the tool call is in `answer.choices[0].message.tool_calls[0]`.
call = answer.choices[0].message.tool_calls[0]
print("Tool call ID:", call.id)
print("Tool name:", call.function.name)
print("Tool arguments:", call.function.arguments)

# In Chat Completions, the function arguments come back as a JSON string
# in `call.function.arguments`, so we parse them before calling `search`.
args = json.loads(call.function.arguments)
print("Parsed args:", args)

result = search(**args)
print("Search result:", result)

# Serialize the search result back to JSON so it can be sent to the model later.
result_json = json.dumps(result, indent=2)
print("Search result JSON:")
print(result_json)

Tool call ID: VUd5JXjb
Tool name: search
Tool arguments: {"query":"can I join the course late"}
Parsed args: {'query': 'can I join the course late'}
Search result: [{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}, {'id': '69d122f12e', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?', 'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list

In [42]:
# In this cell, we append the model's tool call and the tool result to the
# conversation history so the next model request can continue the same turn.
message_history.append(answer.choices[0].message)

# Then we add the tool output as a separate message tied to that tool call.
# The tool call ID links this result to the exact function call the model made.
message_history.append({
    "role": "tool",
    "tool_call_id": call.id,
    "content": result_json,
})

print("Tool result added to message_history.")
print("Message count:", len(message_history))

Tool result added to message_history.
Message count: 8


In [19]:
# Ask the model again with the expanded message history.
# This includes the original question, the model's search request, and the
# search result, which is why it can now answer properly.
response = openai_client.chat.completions.create(
    model=MODEL_ID,
    messages=message_history, # Use `messages=...` in Chat Completions vs `input=...` in Responses API
    tools=[search_tool],
)

# In Chat Completions, the final answer is in `final_answer.choices[0].message.content`.
print(response.choices[0].message.content)

Yes, you can still join the course. However, please note that if you want to receive a certificate, you must submit your project while submissions are still being accepted.


In [20]:
# Token usage helps estimate cost for a tool-using turn.
# The course shows `input_tokens` and `output_tokens` because it uses the Responses API.
# Here, the Gemini-compatible Chat Completions response exposes the same idea with different names below.

# I checked the full response with `JSON(answer.model_dump())` (ABOVE) to see the exact field names.
usage = response.usage

print(usage.completion_tokens) # prompt_tokens = input tokens
print(usage.prompt_tokens) # completion_tokens = output tokens
print(usage.total_tokens) # total_tokens = total for the request

34
763
797


In [21]:
# For each model, the provider publishes a price per million input/output tokens.
# Here we use the token counts from this Gemini-compatible Chat Completions response:
# - prompt_tokens = input tokens
# - completion_tokens = output tokens
# Plug those numbers into the pricing formula to estimate the cost.

def calculate_gemini_price(prompt_tokens, completion_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    prompt_cost = (prompt_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    completion_cost = (completion_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = prompt_cost + completion_cost

    return {
        "prompt_cost": prompt_cost,
        "completion_cost": completion_cost,
        "total_cost": total_cost,
    }

usage = response.usage
result = calculate_gemini_price(usage.prompt_tokens, usage.completion_tokens)

print("Prompt cost: $", round(result["prompt_cost"], 8))
print("Completion cost: $", round(result["completion_cost"], 8))
print("Total cost: $", round(result["total_cost"], 8))

Prompt cost: $ 0.00011445
Completion cost: $ 2.04e-05
Total cost: $ 0.00013485


### Lesson 14 The Agentic Loop
Adding the ability to think, act, observe, and repeat until the task is done

In [23]:
# This developer prompt gives the agent its role and behavior.
# It tells the model when to search, how to expand its queries, and to keep searching
# until it has enough information to answer.
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [25]:
# This helper processes a function call by parsing its JSON arguments,
# running the requested tool, and formatting the tool result so it can be
# sent back to the model in the next step of the loop.

def make_call(call):
    # In Chat Completions, the arguments live under `call.function.arguments`
    # and come back as a JSON string, so we parse them first.
    args = json.loads(call.function.arguments)

    # The tool name is nested under `call.function.name`.
    if call.function.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    # Chat Completions expects the tool result as a `role="tool"` message,
    # linked to the original tool call with `tool_call_id`.
    return {
        "role": "tool",
        "tool_call_id": call.id,
        "content": result_json,
    }

In [45]:
# This cell sends the first request, records the model output, and handles
# any tool calls returned by printing them and appending the tool result
# back into the conversation history.

question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question}
]

response = openai_client.chat.completions.create(
    model=MODEL_ID,
    messages=messages,  # Chat Completions uses `messages=...`, unlike Responses API which uses `input=...`
    tools=[search_tool],
)

# Add the assistant message to the conversation history so the next API call
# includes the latest model output.
messages.append(response.choices[0].message)

has_function_calls = False

# If `tool_calls` is present, the model has not finished answering yet.
# Instead, one or more tools must be run first.
if response.choices[0].message.tool_calls:

    # Loop through each requested tool call.
    # In simple examples there is often only one, but the API can return multiple.
    for item in response.choices[0].message.tool_calls:
        print("function_call:", item.function.name, item.function.arguments)

        # Run the requested tool and package the result into the format
        # expected in the next round.
        call_output = make_call(item)

        # Add the tool result back into the conversation history so the model
        # can read it during the next API call.
        messages.append(call_output)

        # Mark that another API round is still needed after this one.
        has_function_calls = True

# If there are no tool calls, but there is message content, the model
# answered directly and the assistant text can be printed.
elif response.choices[0].message.content:
    print("ASSISTANT:")
    print(response.choices[0].message.content)

function_call: search {"query":"can I join the course late"}


In [46]:
# This is the full agent loop. It keeps calling the model, runs any requested
# tools, and stops when the model returns a message without function calls or
# when the safety cap is reached.

it = 1
max_iterations = 5 # Simple safety cap in case the model does not stop asking for tools.

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.chat.completions.create(
        model=MODEL_ID,
        messages=messages,  # Chat Completions uses `messages=...`, unlike Responses API which uses `input=...`
        tools=[search_tool],
    )
    
    messages.append(response.choices[0].message)

    if response.choices[0].message.tool_calls:
        # The model wants us to run a tool before it can finish answering.
        for item in response.choices[0].message.tool_calls:
            print("function_call:", item.function.name, item.function.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

    elif response.choices[0].message.content:
        # No tool was needed, so the model answered directly in text.
        print("ASSISTANT:")
        print(response.choices[0].message.content)
    
    it = it + 1
    if has_function_calls == False or it > max_iterations:
        break

iteration #1...
ASSISTANT:
Yes, you can still join the course. However, please note that if you want to receive a certificate, you must submit your project while submissions are still being accepted.


In [47]:
# Wrap the full agent loop in a reusable function that takes instructions and a
# question, keeps calling the model, runs any requested tools, and stops when
# the model returns a final answer or when the safety cap is reached.

def agent_loop(instructions, question, max_iterations=5, model=MODEL_ID) -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    last_answer = None  # Stores the final assistant text so the function has a value to return.
    
    it = 1
   
    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.chat.completions.create(
            model=model,
            messages=messages,  # Chat Completions uses `messages=...`, unlike Responses API which uses `input=...`
            tools=[search_tool],
        )

        messages.append(response.choices[0].message)

        if response.choices[0].message.tool_calls:
            for item in response.choices[0].message.tool_calls:
                print("function_call:", item.function.name, item.function.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

        elif response.choices[0].message.content:
            print("ASSISTANT:")
            last_answer = response.choices[0].message.content # Store the final assistant text for return
            print(response.choices[0].message.content)

        it = it + 1
        if has_function_calls == False or it > max_iterations:
            break
    
    return last_answer

agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"how to run Ollama locally"}
iteration #2...
ASSISTANT:
To run Ollama locally, follow these steps:

### 1. Install Ollama
Visit [https://ollama.com/download](https://ollama.com/download) and install it based on your operating system:
*   **macOS**: Download and install the `.pkg` file.
*   **Windows**: Download and install the `.msi` file.
*   **Linux**: Run the following command in your terminal:
    ```bash
    curl -fsSL https://ollama.com/install.sh | sh
    ```

### 2. Start a Model
Once installed, open a terminal and run the following command to download and start the LLaMA 3 model:
```bash
ollama run llama3
```
This will open a chat-like interface where you can interact with the model directly.

### 3. Verify the Server
To test that the Ollama local server is running correctly, execute:
```bash
curl http://localhost:11434
```
You should see a JSON response confirming the server is active.

### 4. Use in Python (Optional)
If you want

'To run Ollama locally, follow these steps:\n\n### 1. Install Ollama\nVisit [https://ollama.com/download](https://ollama.com/download) and install it based on your operating system:\n*   **macOS**: Download and install the `.pkg` file.\n*   **Windows**: Download and install the `.msi` file.\n*   **Linux**: Run the following command in your terminal:\n    ```bash\n    curl -fsSL https://ollama.com/install.sh | sh\n    ```\n\n### 2. Start a Model\nOnce installed, open a terminal and run the following command to download and start the LLaMA 3 model:\n```bash\nollama run llama3\n```\nThis will open a chat-like interface where you can interact with the model directly.\n\n### 3. Verify the Server\nTo test that the Ollama local server is running correctly, execute:\n```bash\ncurl http://localhost:11434\n```\nYou should see a JSON response confirming the server is active.\n\n### 4. Use in Python (Optional)\nIf you want to use Ollama in your Python code, first install the Python client:\n```bas

In [48]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join the course"}
iteration #2...
ASSISTANT:
Yes, you can still join the course. However, please note that if you want to receive a certificate, you must submit your project while submissions are still being accepted.


'Yes, you can still join the course. However, please note that if you want to receive a certificate, you must submit your project while submissions are still being accepted.'

In [49]:
# Adjusted instructions encouraging the agent to make multiple searches before answering.

instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"can I join the course late registration enrollment"}
iteration #2...
ASSISTANT:
Yes, you can still join! You don't need formal registration to get started; you can simply begin learning and submitting homework.

However, please keep a few things in mind regarding the certificate:
*   **Certificates:** To receive a certificate, you must submit your project while submissions are still being accepted.
*   **Self-paced vs. Live:** Certificates are only awarded to those who complete the course with a "live" cohort. This is because the process requires peer-reviewing three capstone projects, which can only be done while the course is actively running.

If you are just looking to learn the material, you are welcome to follow along at your own pace!

Are there any other areas of the course you'd like to explore?


'Yes, you can still join! You don\'t need formal registration to get started; you can simply begin learning and submitting homework.\n\nHowever, please keep a few things in mind regarding the certificate:\n*   **Certificates:** To receive a certificate, you must submit your project while submissions are still being accepted.\n*   **Self-paced vs. Live:** Certificates are only awarded to those who complete the course with a "live" cohort. This is because the process requires peer-reviewing three capstone projects, which can only be done while the course is actively running.\n\nIf you are just looking to learn the material, you are welcome to follow along at your own pace!\n\nAre there any other areas of the course you\'d like to explore?'

In [50]:
# Check whether the agent correctly rejects off-topic questions like chess.
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"what is the queen's gambit in chess"}
iteration #2...
ASSISTANT:
The "Queen's Gambit" is one of the oldest and most popular chess openings. It begins with the moves:

1. **d4 d5**
2. **c4**

In this opening, White offers a pawn (the c4 pawn) to Black, aiming to gain control over the center of the board. It is called a "gambit" because White is effectively sacrificing a pawn in the short term for long-term positional advantages. If Black captures the pawn (2... dxc4), it is known as the "Accepted" version; if Black declines it (often by playing 2... e6), it is the "Declined" version.

It is worth noting that while it is a common chess term, it also gained widespread pop culture recognition as the title of a popular Netflix miniseries centered on chess.

Are there any other areas related to chess or the course that you would like to explore?


'The "Queen\'s Gambit" is one of the oldest and most popular chess openings. It begins with the moves:\n\n1. **d4 d5**\n2. **c4**\n\nIn this opening, White offers a pawn (the c4 pawn) to Black, aiming to gain control over the center of the board. It is called a "gambit" because White is effectively sacrificing a pawn in the short term for long-term positional advantages. If Black captures the pawn (2... dxc4), it is known as the "Accepted" version; if Black declines it (often by playing 2... e6), it is the "Declined" version.\n\nIt is worth noting that while it is a common chess term, it also gained widespread pop culture recognition as the title of a popular Netflix miniseries centered on chess.\n\nAre there any other areas related to chess or the course that you would like to explore?'

In [51]:
# Tighten the instructions so the agent acts like a course assistant and only
# answers from the FAQ, not from general knowledge. This is a soft guardrail.

instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"what's queen gambit"}
iteration #2...
ASSISTANT:
I'm sorry, but that question does not appear to be related to the course or its logistics. Please let me know if there are other areas related to the course that you would like to explore!


"I'm sorry, but that question does not appear to be related to the course or its logistics. Please let me know if there are other areas related to the course that you would like to explore!"

### Lesson 15 ToyAIKit
A small framework for learning, debugging, and moving from handwritten loops to reusable agents

In [75]:
# Setup:
# 1. Install ToyAIKit first: `uv add toyaikit`
# 2. Then import the classes needed for the chat completions workflow.
#    This differs from the course example because it uses the chat
#    completions client and runner.

from toyaikit.llm import OpenAIChatCompletionsClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIChatCompletionsRunner, DisplayingRunnerCallback

print("ToyAIKit chat completions imports loaded successfully.")

ToyAIKit chat completions imports loaded successfully.


In [53]:
# Add the search function to the agent's available tools.
# The schema tells the framework how to expose the tool and what inputs it expects.

agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

print("Search tool registered successfully.")

Search tool registered successfully.


In [55]:
# Define the search tool with a type hint and docstring so ToyAIKit can
# generate the schema automatically.

def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

print("Search function defined with type hints and docstring for automatic schema generation.")

Search function defined with type hints and docstring for automatic schema generation.


In [56]:
# Register the search function directly, without providing a manual schema.
# ToyAIKit can inspect the function's type hint and docstring to generate the
# schema automatically

agent_tools = Tools()
agent_tools.add_tool(search)

print("Search tool registered with automatically generated schema.")

Search tool registered with automatically generated schema.


In [57]:
# Inspect the tool definitions ToyAIKit generated automatically from the
# function's type hint and docstring.

agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

The output is the same JSON schema we wrote by hand in the function-calling lesson, but here ToyAIKit generates it automatically from the function's type hint and docstring.

Modern agent frameworks (e.g., OpenAI Agents SDK, PydanticAI, LangChain and Google ADK ) commonly work this way: you write the tool as a typed Python function, and the framework derives the schema for you.

In [72]:
# Create an OpenAI-compatible client for Gemini, then wrap it in ToyAIKit's
# chat completions client so it can be used by the runner.

import os # Needed to read the Gemini API key from environment variables

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# Raw OpenAI-compatible SDK client, pointed at Gemini
openai_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

# ToyAIKit wrapper around that client
llm_client = OpenAIChatCompletionsClient(
    model=MODEL_ID,
    client=openai_client
)

print("Gemini client configured for ToyAIKit chat completions.")

Gemini client configured for ToyAIKit chat completions.


In [73]:
# Create the chat interface and runner for the notebook.
# This differs slightly from the course example: we use the chat completions
# runner and pass in the Gemini-compatible `llm_client` we created earlier.

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIChatCompletionsRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client
)

print("Chat interface and chat completions runner created with the Gemini-compatible client.")

Chat interface and chat completions runner created with the Gemini-compatible client.


In [74]:
# Run a single prompt through the agent loop.
# The notebook output will show each message and tool call inline.

result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


In [78]:
# Check the cost of the last agent run.
# ToyAIKit likely estimates this from token usage and model pricing. In this notebook,
# the value is only approximate because we're using Gemini through a free-tier
# setup, so it is not the actual billing figure.

result.cost

CostInfo(input_cost=Decimal('0.00035875'), output_cost=Decimal('0.0000825'), total_cost=Decimal('0.00044125'))

In [79]:
# View the full message history captured during the agent run.
# ToyAIKit stores this for us instead of us maintaining the messages list by hand.

result.all_messages

[{'role': 'system',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'How do I run Olama locally?'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='RHY

In [80]:
# Continue the conversation by passing the previous message history into the
# next agent loop call. This lets the model keep the earlier context.

result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [ ]:
# Start ToyAIKit's built-in interactive chat loop for a chat-like notebook
# workflow. Type `stop` when you want to exit.

runner.run()

You: can I get a certificate now?


-> Response received


-> Response received


You: But the course is live?


-> Response received


-> Response received
